# Autonomous Coding Agent via RLVR
Welcome! In this notebook, we will be building a Reinforcement Learning agent using **GRPO** (Group Relative Policy Optimization). We are using **Unsloth** to make sure this fits perfectly.


### Step 1: Install the Required Libraries
First, we need to download the tools that make this possible. We are installing Unsloth for fast training, vLLM for fast generation, and TRL for the Reinforcement Learning loop.

In [1]:
%%capture
!pip install unsloth vllm
!pip install trl peft accelerate bitsandbytes datasets

### Step 2: Load the Model and Tokenizer
We are using `Qwen-2.5-1.5B-Instruct`. It's a very smart model, but small enough to run here. We load it in 4-bit mode to save memory, and we add LoRA adapters so we only have to train a tiny percentage of the model's brain.

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Shrinks the model to fit in RAM
    fast_inference = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-22 01:14:40 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.15.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit with actual GPU utilization = 49.41%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 32.
Unsloth: vLLM's KV Cache can use up to 5.71 

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 02-22 01:15:05 [model.py:541] Resolved architecture: Qwen2ForCausalLM
WARNING 02-22 01:15:05 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 02-22 01:15:05 [model.py:1561] Using max model len 1024
INFO 02-22 01:15:06 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.0.self_attn', 'model.layers.1.mlp', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.7.mlp', 'model.layers.24.mlp', 'model.layers.26.mlp', 'model.layers.15.self_attn'], 'llm_int8_threshold': 6.0}
INFO 02-22 01:15:06 [vllm.py:624] Asynchronous scheduling is

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Step 3: Define the Verifiable Reward Functions

This is the heart of RLVR. We are writing two rules:
1. **Format Reward:** The model gets a point if it uses `<think>` and `<answer>` tags.
2. **Correctness Reward:** The model gets two points if the number inside the `<answer>` tag matches the real answer exactly.

In [3]:
import re

def format_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        if "<think>" in text and "</think>" in text and "<answer>" in text and "</answer>" in text:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def correctness_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    for comp, ans in zip(completions, answer):
        text = comp[0]["content"]
        match = re.search(r"<answer>(.*?)</answer>", text)
        if match and match.group(1).strip() == ans:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

### Step 4: Prep and Format the Dataset
We are grabbing 500 math questions. We have to clean up the data so our reward functions know exactly what number to look for. We also give the model strict instructions on how to behave.

In [4]:
from datasets import load_dataset

# Load 500 questions from the gsm8k dataset
dataset = load_dataset("gsm8k", "main", split="train[:500]")

SYSTEM_PROMPT = """
You are a helpful AI reasoning agent. You must think about the problem step-by-step.
Put all your thinking inside <think> </think> tags.
Put your final, exact answer inside <answer> </answer> tags.
"""

def extract_answer(text):
    parts = text.split("####")
    if len(parts) > 1:
        return parts[1].strip()
    return text.strip()

def format_dataset_for_grpo(example):
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]}
    ]
    correct_answer = extract_answer(example["answer"])

    return {
        "prompt": prompt,
        "answer": correct_answer
    }

# Clean up the whole dataset
dataset = dataset.map(format_dataset_for_grpo)

print("Looks good! First question prepped:")
print(dataset[0]["prompt"])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Looks good! First question prepped:
[{'content': '\nYou are a helpful AI reasoning agent. You must think about the problem step-by-step.\nPut all your thinking inside <think> </think> tags.\nPut your final, exact answer inside <answer> </answer> tags.\n', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]


### Step 5: Start the GRPO Training Loop
We hand our model, our dataset, and our reward functions over to the `GRPOTrainer`. It will start generating answers, scoring itself, and learning from its mistakes.

In [5]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir = "Qwen-GRPO-Agent",
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_prompt_length = 256,
    max_completion_length = 256,
    num_generations = 4, # The model will try 4 answers per question
    max_steps = 100,
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward_func, correctness_reward_func],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


WARNING 02-22 01:24:03 [input_processor.py:287] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,0.000000,0.000000,249.000000,242.000000,256.000000,0.750000,242.000000,242.000000,242.000000,0,0,0,0,0,0.000011,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,187.500000,142.000000,221.000000,0.000000,187.500000,142.000000,221.000000,No Log,No Log,No Log,No Log,No Log,0.000014,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,197.000000,128.000000,256.000000,0.250000,177.333344,128.000000,249.000000,No Log,No Log,No Log,No Log,No Log,0.000012,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,157.000000,126.000000,186.000000,0.000000,157.000000,126.000000,186.000000,No Log,No Log,No Log,No Log,No Log,0.000009,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,238.500000,208.000000,256.000000,0.250000,232.666672,208.000000,251.000000,No Log,No Log,No Log,No Log,No Log,0.000010,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,190.750000,116.000000,242.000000,0.000000,190.750000,116.000000,242.000000,No Log,No Log,No Log,No Log,No Log,0.000012,0.000000,0.000000,0.000000,0.000000
7,0.000100,0.500000,1.000000,255.000000,252.000000,256.000000,0.750000,252.000000,252.000000,252.000000,No Log,No Log,No Log,No Log,No Log,0.000009,0.000000,0.000000,0.500000,1.000000
8,0.000000,0.000000,0.000000,193.750000,154.000000,219.000000,0.000000,193.750000,154.000000,219.000000,No Log,No Log,No Log,No Log,No Log,0.000006,0.000000,0.000000,0.000000,0.000000
9,-0.000100,0.500000,1.000000,206.750000,189.000000,256.000000,0.250000,190.333344,189.000000,193.000000,No Log,No Log,No Log,No Log,No Log,0.000012,0.000000,0.000000,0.500000,1.000000
10,-0.000000,0.500000,1.000000,212.500000,184.000000,236.000000,0.000000,212.500000,184.000000,236.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.500000,1.000000


TrainOutput(global_step=100, training_loss=-5.6934888981308166e-06, metrics={'train_runtime': 2028.6895, 'train_samples_per_second': 0.197, 'train_steps_per_second': 0.049, 'total_flos': 0.0, 'train_loss': -5.6934888981308166e-06})

### Step 6: Save Your Work
Great job! The model is trained. Now we save the final weights. You can also push this directly to your Hugging Face account so you can link it on your resume.

In [6]:
# Save it locally to Colab
model.save_pretrained("my_rlvr_model")
tokenizer.save_pretrained("my_rlvr_model")

print("Model saved successfully!")

Model saved successfully!


In [8]:
# NOTE: To push to Hugging Face, uncomment the lines below and enter your access token when prompted.
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub("Satyamp777/Qwen-GRPO-Agent")
tokenizer.push_to_hub("Satyamp777/Qwen-GRPO-Agent")

README.md:   0%|          | 0.00/584 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Saved model to https://huggingface.co/Satyamp777/Qwen-GRPO-Agent


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...GRPO-Agent/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            